# Create ability groups from the dataset

## Import the dataset

In [ ]:
import pandas as pd
import numpy as np


from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/education-ml-research/ASSISTments2009/skill_builder_data.csv", encoding="latin1")

print("Dataset shape:", df.shape)
print("Students:", df["user_id"].nunique())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_499/2111613670.py:8: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/drive/MyDrive/education-ml-research/ASSISTments2009/skill_builder_data.csv", encoding="latin1")


Dataset shape: (525534, 30)
Students: 4217


## Basic preprocessing of dataset

In [ ]:
ability_df = df[
    ["user_id", "correct"]
].copy()

# Remove missing student IDs and invalid responses
ability_df = ability_df.dropna(subset=["user_id", "correct"])

ability_df["correct"] = pd.to_numeric(
    ability_df["correct"],
    errors="coerce"
)

ability_df = ability_df[
    ability_df["correct"].isin([0, 1])
].copy()

print("Valid interactions:", len(ability_df))
print("Students:", ability_df["user_id"].nunique())

Valid interactions: 525534
Students: 4217


## Calculate overall accuracy for each student

In [ ]:
student_stats = (
    ability_df
    .groupby("user_id")
    .agg(
        total_correct=("correct", "sum"),
        total_attempts=("correct", "count")
    )
)

student_stats["accuracy"] = (
    student_stats["total_correct"] /
    student_stats["total_attempts"]
)

student_stats = student_stats.reset_index()

print(student_stats.head())

   user_id  total_correct  total_attempts  accuracy
0       14             13              51  0.254902
1    21825             21              29  0.724138
2    51933              0               1  0.000000
3    51950              5               6  0.833333
4    52613              4               7  0.571429


## Create ability quartiles

In [ ]:
student_stats["ability_quartile"] = pd.qcut(
    student_stats["accuracy"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

## Verify ability quartiles

In [ ]:
quartile_summary = (
    student_stats
    .groupby("ability_quartile", observed=True)
    .agg(
        students=("user_id", "count"),
        mean_accuracy=("accuracy", "mean"),
        min_accuracy=("accuracy", "min"),
        max_accuracy=("accuracy", "max")
    )
)

print(quartile_summary)

                  students  mean_accuracy  min_accuracy  max_accuracy
ability_quartile                                                     
Q1                    1056       0.266387      0.000000      0.490196
Q2                    1053       0.575853      0.490566      0.663928
Q3                    1056       0.733381      0.666667      0.800000
Q4                    1052       0.914464      0.800595      1.000000


## Save the dataset

In [ ]:
import os

output_path = (
    "/content/drive/MyDrive/education-ml-research/"
    "ASSISTments2009/student_ability_quartiles.csv"
)

student_stats.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"Rows: {len(student_stats)}")

Saved to: /content/drive/MyDrive/education-ml-research/ASSISTments2009/student_ability_quartiles.csv
Rows: 4217
